In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import pandas as pd
from sklearn.metrics import classification_report

def main():
    # 1. Configuration des transformations d'images pour ResNet-50
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    print("Chargement du dataset CelebA (cela peut prendre un moment lors du premier téléchargement)...")
    
    # 2. Chargement de CelebA via torchvision
    # Remarque : si le téléchargement automatique pose problème, vous pouvez placer les fichiers dans './data'
    train_dataset = datasets.CelebA(
        root='./data', 
        split='train', 
        target_type=['attr'], 
        transform=transform, 
        download=True
    )
    
    train_dataset = torch.utils.data.Subset(train_dataset, range(10000))

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

    # 3. Chargement du modèle ResNet-50 pré-entraîné
    print("Initialisation de ResNet-50...")
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)

    # Adaptation de la dernière couche pour une classification binaire (ex: attribut cible index 31 : Smiling)
    # CelebA contient 40 attributs dans son tenseur 'attr'
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)  # 2 classes : Non / Oui

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # 4. Définition de la fonction de perte et de l'optimiseur
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    # 5. Entraînement rapide 
    print(f"Début de l'entraînement sur {device}...")
    model.train()
    for epoch in range(1): 
        running_loss = 0.0
        for i, (images, attrs) in enumerate(train_loader):
            images = images.to(device)
            
            # Sélection de l'attribut cible (ex: index 31 pour 'Smiling')
            # Vous pouvez inspecter les indices des attributs dans train_dataset.dataset.attr_names
            targets = attrs[:, 31].to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 50 == 49:
                print(f"[Époque {epoch+1}, Batch {i+1}] Perte (Loss): {running_loss / 50:.3f}")
                running_loss = 0.0

    print("Entraînement initial terminé avec succès !")
    
    # Sauvegarde du modèle de base
    torch.save(model.state_dict(), "resnet50_celeba_base.pth")
    print("Modèle sauvegardé sous 'resnet50_celeba_base.pth'.")

if __name__ == "__main__":
    main()

Chargement du dataset CelebA (cela peut prendre un moment lors du premier téléchargement)...


RuntimeError: To download files from GDrive, 'gdown' is required. You can install it with 'pip install gdown'.